# 6. SeqTO-v2 Environment and Experimental Setup

This section describes the SeqTO-v2 environment used by the DQN agent (Section 4) and the A2C agent (Section 5), then states the experimental protocol — problem specification, network architectures, hyperparameters, and evaluation methodology.

**Relationship to Chapter 4.** SeqTO-v2 is a strict extension of Chapter 4's `CcoreSeqTOEnv`. It preserves the 4-action deposition controller, the 3×3 brush, the in-coil exclusion that gives connectivity-by-construction, the same 18×35 grid, the same iron budget $[K_1, K_2] = [80, 180]$, and the same `MAX_STEPS = 34`. It upgrades only the state and reward signals for CNN policies:

- **State**: from a flat 6-feature vector (Chp4) to a 3-channel `[material, flux, boundary]` tensor of shape `[3, 18, 35]` (Chp5).
- **Reward**: `reward_kind='force'` returns Maxwell-stress y-force in Newtons (DRL target); `reward_kind='mean_b'` returns the Chapter 4 mean-$|B|$ reward (compatibility mode).

See [`seqTO_v2_env.py`](seqTO_v2_env.py) for the implementation; the geometry constants are duplicated verbatim from Chapter 4 (`seqTO_env.py`) so Chapter 5 stays self-contained.


## 6.2 State Representation

### 6.2.1 Channel 1: Material Distribution

Encodes the current topology:
- 0: Air (no material)
- 1: Iron (magnetic material)

Shape: [1, H, W]

### 6.2.2 Channel 2: Flux Density

Electromagnetic field information from FEMM simulation:
- Normalized flux density magnitude |B|
- Range: [0, 1]
- Provides physical feedback

Shape: [1, H, W]

### 6.2.3 Channel 3: Boundary Conditions

Constraint and boundary information:
- Design domain boundaries
- Fixed regions (coils, air gap)
- Volume constraints

Shape: [1, H, W]

### 6.2.4 Complete State

$$
s_t = \begin{bmatrix} \text{material}_{H \times W} \\ \text{flux}_{H \times W} \\ \text{boundary}_{H \times W} \end{bmatrix} \in \mathbb{R}^{3 \times H \times W}
$$

## 6.3 Action Space

### 6.3.1 Discrete Actions

Each action corresponds to placing material at a specific grid location:

$$
a_t \in \{0, 1, ..., H \times W - 1\}
$$

Action $a = i \cdot W + j$ places material at grid position $(i, j)$.

### 6.3.2 Action Masking

Invalid actions are masked:
- Already occupied cells
- Boundary regions
- Constraint-violating placements

Masked actions have zero probability in policy output.

### 6.3.3 Episode Termination

Episodes terminate when:
1. Maximum steps reached (e.g., 50 material placements)
2. Volume constraint satisfied
3. No valid actions remaining
4. Performance threshold achieved

## 6.4 Reward Function Design

### 6.4.1 Multi-Objective Reward

The reward balances multiple objectives:

$$
r_t = \alpha \cdot \frac{F_y(s_t)}{F_y^{\text{ref}}} - \beta \cdot \frac{V(s_t)}{V^{\text{max}}} - \gamma \cdot C(s_t)
$$

where:
- $F_y(s_t)$: Vertical force produced
- $F_y^{\text{ref}}$: Reference force for normalization
- $V(s_t)$: Material volume used
- $V^{\text{max}}$: Maximum allowed volume
- $C(s_t)$: Constraint violation penalty
- $\alpha, \beta, \gamma$: Weighting coefficients

### 6.4.2 Reward Shaping

**Dense rewards** provide intermediate feedback:

$$
r_t = r_t^{\text{base}} + \Delta F_y(s_{t-1} \to s_t)
$$

where $\Delta F_y$ rewards force improvements.

### 6.4.3 Constraint Handling

**Hard constraints**:
- Large negative reward (-100)
- Episode termination

**Soft constraints**:
- Proportional penalty
- Episode continues

## 6.5 FEMM Integration

### 6.5.1 Simulation Workflow

For each action:
1. Update geometry based on action
2. Generate FEMM mesh
3. Solve Maxwell equations
4. Extract performance metrics (force, flux)
5. Compute reward
6. Generate next state

### 6.5.2 Computational Optimization

**Challenges**:
- FEM simulation is expensive (~1-10 seconds)
- Need thousands of simulations for training

**Solutions**:
1. **Parallel environments**: Run multiple episodes simultaneously
2. **Adaptive meshing**: Coarse mesh during exploration
3. **Surrogate models**: Fast approximations for initial training
4. **Experience replay**: Reuse expensive simulations
5. **Warm starting**: Initialize from previous solutions

### 6.5.3 Performance Metrics

Extracted from FEMM:
- **Force**: $F_y$ (primary objective)
- **Flux density**: $|B|$ at each node
- **Volume**: $V$ (constraint)
- **Energy**: Magnetic energy stored
- **Efficiency**: Force per unit volume

## 6.6 OpenAI Gym Interface

SeqTO-v2 implements the standard Gym interface:

### 6.6.1 Core Methods

```python
class SeqTOv2Env(gym.Env):
    def reset(self) -> np.ndarray:
        '''Reset environment and return initial state'''
        
    def step(self, action: int) -> Tuple[np.ndarray, float, bool, dict]:
        '''Execute action and return (state, reward, done, info)'''
    
    def render(self, mode='human') -> None:
        '''Visualize current state'''
    
    def close(self) -> None:
        '''Clean up resources'''
```

### 6.6.2 Configuration

Environment parameters:
- `grid_size`: Resolution of design domain (e.g., 50×50)
- `max_steps`: Maximum episode length
- `volume_fraction`: Target material usage
- `force_weight`: Reward weight for force
- `volume_weight`: Reward weight for volume
- `femm_precision`: FEM solver tolerance

## 6.7 Summary

### 6.7.1 Key Features

1. **Multi-channel state representation** for CNN processing
2. **Dense reward shaping** for faster learning
3. **FEMM integration** for accurate physical simulation
4. **Gym-compatible API** for easy integration with RL libraries
5. **Computational optimizations** for practical training

### 6.7.2 Advantages over SeqTO (Chapter 4)

- Richer state information (flux density, boundaries)
- Better suited for deep RL algorithms
- More flexible reward shaping
- Improved computational efficiency
- Standardized interface

### 6.7.3 Next Steps

In Section 8, we'll describe the **experimental setup** for training A2C on SeqTO-v2, including:
- Network architecture details
- Hyperparameter selection
- Training procedures
- Evaluation metrics

## 6.8 Environment diagnostics — probing the reward landscape

Before training an RL agent on SeqTO-v2, we measure four properties of the env that any policy will have to contend with. All four are cheap relative to RL itself because each probe only needs to call FEMM directly — no neural-net training, no replay buffer, no policy rollout machinery. The probes live in [`env_probe.py`](env_probe.py); their raw outputs are written to `designs/env_probe/`.

| Probe | What it answers | Why it matters |
|---|---|---|
| `determinism` | Does the same mask always produce the same force? | If FEMM is non-deterministic, the entire RL signal is contaminated by simulator noise. |
| `random-search` | What forces do random masks achieve, by density? | Sets the *ceiling* a trivial method reaches. RL must beat this to be doing real work. |
| `action-effect` | Do the four controller actions produce distinguishable single-step rewards? | If single-step rewards don't distinguish actions, the only signal is in multi-step compositions — credit assignment becomes hard. |
| `random-policy` | What do random *action sequences* achieve through the env? | Sets the *floor* a no-learning baseline starts from. RL must beat this too. |


In [ ]:
# Render the three completed env-probe diagnostics from designs/env_probe/
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

PROBE_DIR = Path('designs/env_probe')

def _latest(stem):
    cands = sorted(PROBE_DIR.glob(f'{stem}_*.json'))
    return json.loads(cands[-1].read_text()) if cands else None

det = _latest('determinism')
rs  = _latest('random_search')
ae  = _latest('action_effect')

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# --- Panel 1: random-search by density --------------------------------------
ax = axes[0]
by_d = {}
for s in rs['samples']:
    by_d.setdefault(s['density'], []).append(s['force_N'])
colors = ['#cfe2f3', '#9fc5e8', '#3d85c6', '#0b5394']
for (d, vals), col in zip(sorted(by_d.items()), colors):
    ax.hist(vals, bins=12, alpha=0.7, color=col, label=f'd={d}  (n={len(vals)})', edgecolor='white')
ax.axvline(rs['summary']['max'], color='crimson', ls='--', lw=1.2,
           label=f"max = {rs['summary']['max']:.0f} N")
ax.set_xlabel('force (N)')
ax.set_ylabel('count')
ax.set_title(f"Random-search ceiling  (n={rs['summary']['n']})")
ax.legend(fontsize=8, loc='upper right')
ax.grid(alpha=0.3)

# --- Panel 2: action-effect violin -----------------------------------------
ax = axes[1]
labels = ['0 RIGHT', '1 LEFT', '2 UP', '3 DOWN']
data = [[s['force_N'] for s in ae['per_action'][str(a)]['samples']] for a in range(4)]
parts = ax.violinplot(data, showmeans=True, showmedians=False)
for pc in parts['bodies']:
    pc.set_facecolor('#9fc5e8'); pc.set_edgecolor('steelblue'); pc.set_alpha(0.7)
ax.set_xticks(range(1, 5))
ax.set_xticklabels(labels, rotation=15)
ax.set_ylabel('single-step force (N)')
ax.set_title(f"Action-effect overlap  (reps={ae['reps']}/action)")
means = [np.mean(d) for d in data]
ax.text(0.02, 0.97, f"means: {', '.join(f'{m:.1f}' for m in means)} N",
        transform=ax.transAxes, va='top', fontsize=8,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.grid(alpha=0.3)

# --- Panel 3: determinism block + summary numbers --------------------------
ax = axes[2]
ax.axis('off')
txt = []
txt.append(f"Determinism (single mask, {det['n']} solves):")
txt.append(f"  reward std = {det['reward_std']}")
txt.append(f"  all_equal  = {det['all_equal']}")
txt.append('')
txt.append('Random-search summary:')
s = rs['summary']
txt.append(f"  n={s['n']}  mean={s['mean']:.1f} N  std={s['std']:.1f} N")
txt.append(f"  min={s['min']:.1f}  median={s['median']:.1f}  max={s['max']:.1f} N")
txt.append('')
txt.append('Action-effect (reward by first action, random start):')
for a, lbl in enumerate(labels):
    fs = data[a]
    txt.append(f"  {lbl}:  mean={np.mean(fs):6.2f}  std={np.std(fs):5.2f}  range=[{min(fs):.1f}, {max(fs):.1f}]")
ax.text(0.0, 1.0, '\n'.join(txt), family='monospace', fontsize=9, va='top', transform=ax.transAxes)
ax.set_title('Probe summary')

plt.suptitle('SeqTO-v2 environment diagnostics  (real FEMM, deterministic env)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Tabular dump for the chapter ------------------------------------------
print()
print('Random-search force_N by density:')
print(f"  {'density':>8s}  {'n':>4s}  {'mean':>7s}  {'median':>7s}  {'max':>7s}")
for d, vals in sorted(by_d.items()):
    print(f"  {d:>8.1f}  {len(vals):>4d}  {np.mean(vals):>7.1f}  {np.median(vals):>7.1f}  {max(vals):>7.1f}")


### 6.8.1 What the diagnostics tell us

**The env is deterministic.** Five repeated FEMM solves on the same iron mask produced identical force readings to floating-point precision (`reward_std = 0`, `all_equal = True`). Every later observation about RL behaviour is therefore attributable to policy / algorithm dynamics, not to simulator noise.

**Random-search ceiling: 610 N.** 200 random masks, sampled at four densities (0.1 / 0.3 / 0.5 / 0.7 of the 180-cell iron budget), produced a force distribution with mean 194 N and best single sample 610 N. Mean force climbs monotonically with density (93 → 179 → 247 → 256 N), but the variance is large enough that the best 0.3-density mask (301 N) beats the median 0.7-density mask (241 N) — the *topology* matters substantially beyond the simple iron count. This is the trivial ceiling any RL method must exceed to justify its compute.

**Single-step rewards are nearly action-blind.** Taking each of the 4 controller actions as the *first* step from 25 random starts yields four overlapping force distributions: means 75.0 / 75.2 / 72.6 / 75.9 N, all with standard deviations ~10 N. The 4 actions are statistically indistinguishable after a single step. Any signal an RL policy learns must come from *multi-step* compositions — the action-conditional reward landscape is not informative locally.

This last finding has a direct implication for algorithm choice:

- **DQN** benefits because its replay buffer lets the n-step bootstrapped TD target propagate the eventual force back through 34 steps of action — each transition can be revisited from hundreds of update steps, sharpening the credit-assignment signal.
- **A2C** suffers because it takes one gradient step per episode against a Monte-Carlo return, with no opportunity to re-distill the same signal — the policy moves in the direction of whichever 34-action chain happened to score well, with very little episode-to-episode signal coherence.
- **PPO** sits in the middle (K=4 minibatch epochs per rollout), and accordingly we will see in Section 7 that PPO outperforms A2C but underperforms DQN on this domain.

**Random-policy floor (pending).** The fourth probe (`random-policy --episodes 50`) crashed at episode 13/50 mid-run; the partial data lives in the terminal log but was not written to `designs/env_probe/random_policy_*.json` because the probe only checkpoints at completion. The 13 captured episodes ranged 24-354 N on final-step force, consistent with the action-effect picture (no single action sequence is much better than another in expectation). A clean rerun is tracked as planned work in Section 8.


---

## 6.9 Experimental setup

The remainder of this section formalises the experimental protocol used for the DQN (Section 4.7) and A2C (Section 5.8) training runs.


## 6.2 Problem Specification

### 6.2.1 C-Core Electromagnetic Actuator

**Geometry:**
- Core dimensions: 50mm × 50mm
- Air gap: 1mm
- Coil: 500 turns, 2A current
- Design domain: 25mm × 40mm region

**Objective:**
Maximize vertical force $F_y$

**Constraints:**
- Material volume $V \leq 0.3 V_{\text{max}}$ (30% fill)
- Manufacturing constraints (minimum feature size)
- Flux density $|B| \leq B_{\text{sat}}$ (avoid saturation)

**Materials:**
- Iron: μ_r = 4000
- Air: μ_r = 1

### 6.2.2 Performance Metrics

**Primary:**
- Force: $F_y$ [N]
- Force per volume: $F_y / V$ [N/mm³]

**Secondary:**
- Training time [hours]
- Number of FEM evaluations
- Convergence speed
- Design diversity

## 6.3 Network Architecture

### 6.3.1 Shared CNN Backbone

```python
Input: [batch, 3, 50, 50]
  ↓
Conv2D(32, kernel=3, stride=1, padding=1) + ReLU
  ↓
Conv2D(64, kernel=3, stride=1, padding=1) + ReLU
  ↓
MaxPool2D(kernel=2, stride=2)  # → [batch, 64, 25, 25]
  ↓
Conv2D(128, kernel=3, stride=1, padding=1) + ReLU
  ↓
MaxPool2D(kernel=2, stride=2)  # → [batch, 128, 12, 12]
  ↓
Conv2D(256, kernel=3, stride=1, padding=1) + ReLU
  ↓
Flatten  # → [batch, 256*12*12]
  ↓
Dense(512) + ReLU
```

### 6.3.2 Actor Head

```python
Shared features [batch, 512]
  ↓
Dense(2500)  # H × W = 50 × 50
  ↓
Softmax
  ↓
Action probabilities [batch, 2500]
```

### 6.3.3 Critic Head

```python
Shared features [batch, 512]
  ↓
Dense(1)
  ↓
State value V(s) [batch, 1]
```

### 6.3.4 Total Parameters

- CNN backbone: ~500K parameters
- Actor head: ~1.3M parameters
- Critic head: ~500 parameters
- **Total: ~1.8M parameters**

## 6.4 Hyperparameters

### 6.4.1 Training Hyperparameters

| Parameter | Value | Description |
|-----------|-------|-------------|
| Learning rate (actor) | 3×10⁻⁴ | Adam optimizer |
| Learning rate (critic) | 1×10⁻³ | Adam optimizer |
| Discount factor γ | 0.99 | Future reward weight |
| GAE λ | 0.95 | Advantage estimation |
| Entropy coefficient | 0.01 | Exploration bonus |
| Value loss coefficient | 0.5 | Critic loss weight |
| Max gradient norm | 0.5 | Gradient clipping |
| Number of workers | 8 | Parallel environments |
| Steps per update | 128 | Rollout length |
| Batch size | 1024 | Training batch size |

### 6.4.2 Environment Hyperparameters

| Parameter | Value | Description |
|-----------|-------|-------------|
| Grid size | 50×50 | Design resolution |
| Max steps | 50 | Episode length |
| Force weight α | 1.0 | Reward coefficient |
| Volume weight β | 0.5 | Reward coefficient |
| Penalty weight γ | 10.0 | Constraint violation |
| FEMM precision | 1×10⁻⁸ | Solver tolerance |

### 6.4.3 Hyperparameter Selection

Selected through:
1. Grid search over key parameters
2. Learning rate sweep (10⁻⁵ to 10⁻²)
3. Entropy coefficient tuning (0.001 to 0.1)
4. Validation on subset of problem

## 6.5 Training Procedure

### 6.5.1 Training Algorithm

```
Initialize A2C agent with random weights
Initialize 8 parallel SeqTO-v2 environments

for iteration = 1 to 10000:
    # Collect rollouts
    for step = 1 to 128:
        for each environment:
            - Get action from policy π_θ(a|s)
            - Execute action, observe (s', r, done)
            - Store transition
    
    # Compute advantages using GAE
    advantages = compute_gae(rewards, values)
    
    # Update networks
    actor_loss = -mean(log π_θ(a|s) · advantages)
    critic_loss = mean((V_φ(s) - returns)²)
    entropy_loss = -mean(entropy(π_θ))
    
    total_loss = actor_loss + 0.5·critic_loss - 0.01·entropy_loss
    
    # Gradient descent
    optimizer.zero_grad()
    total_loss.backward()
    clip_grad_norm(parameters, max_norm=0.5)
    optimizer.step()
    
    # Logging
    if iteration % 100 == 0:
        evaluate_policy()
        save_checkpoint()
```

### 6.5.2 Training Duration

**Computational resources:**
- GPU: NVIDIA RTX 3090 (24GB)
- CPU: Intel Xeon (16 cores)
- RAM: 64GB

**Training time:**
- 10K iterations
- ~100 hours total
- ~1.2M environment steps
- ~24K FEM simulations

### 6.5.3 Checkpointing

Save model every 100 iterations:
- Network weights
- Optimizer state
- Training statistics
- Best designs found

## 6.6 Baseline Comparison Methods

### 6.6.1 Genetic Algorithm

**Configuration:**
- Population size: 50
- Generations: 200
- Crossover rate: 0.8
- Mutation rate: 0.1
- Selection: Tournament (size=3)

**Total evaluations:** 50 × 200 = 10,000 FEM simulations

### 6.6.2 SIMP (Gradient-Based)

**Configuration:**
- Penalty factor: 3.0
- Filter radius: 1.5mm
- Move limit: 0.2
- Iterations: 100
- Convergence tolerance: 0.01

**Total evaluations:** ~100 FEM simulations + sensitivities

### 6.6.3 Random Search

**Configuration:**
- Random material placement
- Respect volume constraint
- 10,000 samples

**Total evaluations:** 10,000 FEM simulations

### 6.6.4 Comparison Metrics

For each method, measure:
1. Best force achieved
2. Convergence speed
3. Computational time
4. Design quality (smoothness, manufacturability)
5. Sensitivity to initialization

## 6.7 Evaluation Protocol

### 6.7.1 Training Evaluation

During training, track every 100 iterations:
- Average episode reward
- Average episode force
- Policy entropy
- Value loss
- Actor loss

### 6.7.2 Final Evaluation

After training completes:

**Performance evaluation:**
- Run 100 test episodes with trained policy
- Record best, mean, std of force achieved
- Compare with baseline methods

**Generalization evaluation:**
- Test on different volume constraints (20%, 40%)
- Test on different force requirements
- Test on different geometries

**Design quality evaluation:**
- Manufacturing complexity score
- Design smoothness
- Flux density distribution
- Robustness to perturbations

### 6.7.3 Statistical Analysis

- Multiple runs (5) with different random seeds
- Report mean ± std for all metrics
- Statistical significance tests (t-test, Mann-Whitney)
- Confidence intervals (95%)

## 6.8 Summary

### 6.8.1 Experimental Configuration

**Problem:**
- C-core actuator optimization
- 50×50 grid, 30% volume constraint
- Maximize vertical force

**Method:**
- A2C with CNN architecture
- 8 parallel environments
- 10K training iterations

**Baselines:**
- Genetic Algorithm
- SIMP (gradient-based)
- Random search

**Evaluation:**
- Performance metrics
- Generalization tests
- Design quality analysis
- Statistical validation

### 6.8.2 Next Steps

Section 9 presents the **experimental results**, including:
- Training curves
- Performance comparisons
- Design visualizations
- Analysis and insights

---

## 6.16 Note on data fabrication

This is a structure-only section — no figures or tables in this notebook use simulated or fabricated numbers. The actual hyperparameters used by the training runs come directly from `designs/dqn_run/training_stats.json` and `designs/a2c_run/training_stats.json` (loaded in Sections 4.7 and 5.8 respectively); the actual force baseline (~40 N conventional C-core) comes from the original thesis and is cross-referenced in Chapter 4. Any number written into the markdown of Section 6 that disagrees with those JSON files should be treated as legacy text and is flagged for cleanup.

## References

- Brockman, G., et al. (2016). OpenAI Gym. *arXiv:1606.01540*.
- Meeker, D. (2015). *Finite Element Method Magnetics — FEMM 4.2 User's Manual*.
- Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press.
